## Sagnac Frequency - Backscatter Correction - Compare Setups

Explore the longterm behaviour of the RV sagnac beat

## Imports

In [2]:
import os
import gc
import matplotlib.pyplot as plt
import numpy as np

from datetime import datetime, date
from pandas import DataFrame, read_pickle, date_range, concat, read_csv
from obspy import UTCDateTime, read, Trace, Stream, read_inventory
from scipy.signal import hilbert
from scipy.ndimage import gaussian_filter1d


import sys
sys.path.append("../")

from functions.get_fft import __get_fft
from functions.multitaper_psd import __multitaper_psd
from functions.welch_psd import __welch_psd
from functions.rotate_romy_ZUV_ZNE import __rotate_romy_ZUV_ZNE
from functions.reduce import __reduce
from functions.load_backscatter_data import __load_backscatter_data
from functions.backscatter_correction import __backscatter_correction

from andbro__read_sds import __read_sds
from andbro__readYaml import __readYaml
from andbro__load_FURT_stream import __load_furt_stream

In [3]:
if os.uname().nodename == 'lighthouse':
    root_path = '/home/andbro/'
    data_path = '/home/andbro/kilauea-data/'
    archive_path = '/home/andbro/freenas/'
    bay_path = '/home/andbro/ontap-ffb-bay200/'
    lamont_path = '/home/andbro/lamont/'
elif os.uname().nodename == 'kilauea':
    root_path = '/home/brotzer/'
    data_path = '/import/kilauea-data/'
    archive_path = '/import/freenas-ffb-01-data/'
    bay_path = '/import/ontap-ffb-bay200/'
    lamont_path = '/lamont/'
elif os.uname().nodename in ['lin-ffb-01', 'ambrym', 'hochfelln']:
    root_path = '/home/brotzer/'
    data_path = '/import/kilauea-data/'
    archive_path = '/import/freenas-ffb-01-data/'
    bay_path = '/import/ontap-ffb-bay200/'
    lamont_path = '/lamont/'

## Configurations

In [4]:
config = {}

config['ring'] = "Z"

# config['delta'] = 120

# switch of PDs for monobeams at RZ & new control loop
config['tbeg'] = UTCDateTime("2024-10-23 00:00")
# config['tend'] = UTCDateTime("2024-08-10 00:00")
flim1, flim2 = 553.569, 553.577

# path to Sagnac data
config['path_to_data'] = data_path+"sagnac_frequency/data/compare_methods/backscatter/"

# config['path_to_out_data'] = data_path+"sagnac_frequency/data/"

# path to out figures
config['path_to_figs'] = data_path+"sagnac_frequency/figures/"


In [5]:
class sagnacf:

    import numpy as np
    from obspy import UTCDateTime

    def __init__(self, delta=60, ring="Z"):

        self.delta = delta

        self.ring = ring

    def load_data(self, path, filename):
        self.data = read_pickle(path+filename)

    def relative_timing(self, tbeg):
        self.data['time_sec'] = self.data.time1 - UTCDateTime(tbeg) + self.delta/2

    def trim(self, tbeg=None, tend=None):
        if tbeg is not None:
            _df = self.data
            self.data = _df[_df.time1 >= UTCDateTime(tbeg)]
        if tend is not None:
            _df = self.data
            self.data = _df[_df.time2 <= UTCDateTime(tend)]

    def apply_backscatter_correction(self):

        # copy data
        _data = self.data

        # compute unwrapped phase difference
        phase_difference = np.unwrap(_data.f1_phw) - np.unwrap(_data.f2_phw)

        # compute backscatter correction
        _data['fj_bs'], _, _ = self.bs_correction(_data.f1_ac / _data.f1_dc,
                                                  _data.f2_ac / _data.f2_dc,
                                                  phase_difference,
                                                  _data.fj_fs,
                                                  np.nanmedian(_data.fj_fs),
                                                  cm_filter_factor=1.033,
                                                 )
        # reasign updated dataframe
        self.data = _data

    def get_scalefactor(self, output):

        from numpy import pi, sqrt, arccos, deg2rad, arcsin, cos, sin, array, zeros

        # angle in horizontal plane
        h_rot = {"Z":0, "U":0, "V":60, "W":60}

        # angle from vertical
        v_rot = {"Z":0, "U":109.5, "V":70.5, "W":70.5}
        # v_rot = {"Z":-90, "U":19.5, "V":-19.5, "W":-19.5}

        # side length
        L = {"Z":11.2, "U":12, "V":12, "W":12}

        # wavelength
        lamda = 632.8e-9

        # Scale factor
        S = (sqrt(3)*L[self.ring])/(3*lamda)

        # ROMY latitude
        lat = deg2rad(48.162941)
        lon = deg2rad(11.275501)

        # nominal Earth rotation
        omegaE = 2*pi/86400 * array([0, 0, 1])

        # matrix 1
        D = array([[-sin(lat)*cos(lon), -sin(lon), cos(lat)*cos(lon)],
                   [sin(lat)*sin(lon), cos(lon), cos(lat)*sin(lon)],
                   [cos(lat), 0, sin(lat)]
                  ])

        # tilt
        da = deg2rad(0)
        dz = deg2rad(0)

        # tilt matrix
        R = array([[1, -da, -dz], [da,  1, 0], [dz, 0, 1]])

        pv = deg2rad(v_rot[self.ring])
        ph = deg2rad(h_rot[self.ring])

        # normal vector of ring
        nx = array([[sin(pv)*cos(ph)], [sin(pv)*sin(ph)], [cos(pv)]])

        one = array([0, 0, 1])

        out = S * ( one @ ( D @ (R @ nx) ) )[0]

        self.get_scalefactor = out

        if output:
            return out

    @staticmethod
    def sagnac_to_tilt(data, ring="Z", tilt="n-s"):

        from numpy import pi, sqrt, arccos, deg2rad, arcsin, cos, sin, array, zeros

        # angle in horizontal plane
        h_rot = {"Z":0, "U":0, "V":60, "W":60}

        # angle from vertical
        v_rot = {"Z":0, "U":109.5, "V":70.5, "W":70.5}
        # v_rot = {"Z":-90, "U":19.5, "V":-19.5, "W":-19.5}

        # side length
        L = {"Z":11.2, "U":12, "V":12, "W":12}

        # Scale factor
        S = (sqrt(3)*L[ring])/(3*632.8e-9)

        # ROMY latitude
        lat = deg2rad(48.162941)
        lon = deg2rad(11.275501)

        # nominal Earth rotation
        omegaE = 2*pi/86400 * array([0, 0, 1])

        # matrix 1
        D = array([[-sin(lat)*cos(lon), -sin(lon), cos(lat)*cos(lon)],
                   [sin(lat)*sin(lon), cos(lon), cos(lat)*sin(lon)],
                   [cos(lat), 0, sin(lat)]
                  ])

        # tilt
        da = deg2rad(0)
        dz = deg2rad(0)

        # tilt matrix
        R = array([[1, -da, -dz], [da,  1, 0], [dz, 0, 1]])

        pv = deg2rad(v_rot[ring])
        ph = deg2rad(h_rot[ring])

        # normal vector of ring
        nx = array([[sin(pv)*cos(ph)], [sin(pv)*sin(ph)], [cos(pv)]])

        # terms
        # term1 = cos(v_rot[ring])*sin(lat)
        # term2 = cos(lat)*sin(v_rot[ring])*cos(h_rot[ring])
        term1 = cos(pv)*sin(lat)
        term2 = cos(lat)*sin(pv)*cos(ph)

        # tilt factor
        # fz = sin(lat)*sin(v_rot[ring])*cos(h_rot[ring]) - cos(v_rot[ring])*cos(lat)
        # fa = sin(v_rot[ring])*sin(h_rot[ring])*cos(lat)
        fz = sin(lat)*sin(pv)*cos(ph) - cos(pv)*cos(lat)
        fa = sin(pv)*sin(ph)*cos(lat)

        if tilt == "n-s":
            out = ( (data /S /omegaE[2]) - term1 - term2 ) / fz
        elif tilt == "e-w":
            out = ( (data /S /omegaE[2]) - term1 - term2 ) / fa
        elif tilt == "tilt_to_hz":
            out = zeros(len(data))
            for n in range(len(data)):
                out[n] = S * ( (omegaE + data[n]) @ ( D @ (R @ nx) ) )

        return out

    @staticmethod
    def bs_correction(m01, m02, phase0, w_obs, fs0, cm_filter_factor=1.033):

        from numpy import array, sin, cos

        # Correct for bias
        m1 = m01 * ( 1 + m01**2 / 4 )
        m2 = m02 * ( 1 + m02**2 / 4 )

        # angular correction for phase
        phase = phase0 + 0.5 * m1 * m2 * sin( phase0 )

        # compute squares of common-mode modulations
        m2c = ( m1**2 + m2**2 + 2*m1*m2*cos( phase ) ) / 4

        # compute squares of differential-mode modulations
        m2d = ( m1**2 + m2**2 - 2*m1*m2*cos( phase ) ) / 4  ## different angle!

        # correct m2c for gain saturation of a HeNe laser
        # m2c = m2c * ( 1 + ( beta + theta )**2 * fL**2 * I0**2 / ws**2 )
        m2c = m2c * cm_filter_factor

        # compute backscatter correction factor
        M = m2c - m2d + 0.25 * m1**2 * m2**2 * sin(phase)**2

        # correction term
        term = ( 4 + M ) / ( 4 - M )

        # backscatter correction
        correction = -1 * ( term - 1 ) * fs0
        # w_corrected = np.array(w_obs) + correction

        # apply backscatter correction
        w_corrected = array(w_obs) * term

        return w_corrected, correction, term


### Load backscatter data

In [6]:
lbl1 = "Z_T120_hilbert"

bs1 = sagnacf(delta=120)

bs1.load_data(config['path_to_data'], "FJZ_20241023_backscatter_T120_hilbert.pkl")

bs1.relative_timing(config['tbeg'])

bs1.apply_backscatter_correction()

bs1.trim(tbeg=UTCDateTime("2024-10-23 12:00"))

In [7]:
lbl2 = "Z_T60_hilbert"

bs2 = sagnacf(delta=60)

bs2.load_data(config['path_to_data'], "FJZ_20241023_backscatter_T60_hilbert.pkl")

bs2.relative_timing(config['tbeg'])

bs2.apply_backscatter_correction()

bs2.trim(tbeg=UTCDateTime("2024-10-23 12:00"))

In [8]:
lbl3 = "Z_T120_hilbert_ca"

bs3 = sagnacf(delta=120)

bs3.load_data(config['path_to_data'], "FJZ_20241023_backscatter_T120_hilbert_CA.pkl")

bs3.relative_timing(config['tbeg'])

bs3.apply_backscatter_correction()

bs3.trim(tbeg=UTCDateTime("2024-10-23 12:00"))

In [11]:
lbl4 = "Z_T60_hilbert_ca"

bs4 = sagnacf(delta=60)

bs4.load_data(config['path_to_data'], "FJZ_20241023_backscatter_T60_hilbert_CA.pkl")

bs4.relative_timing(config['tbeg'])

bs4.apply_backscatter_correction()

bs4.trim(tbeg=UTCDateTime("2024-10-23 12:00"))

In [12]:
lbl5 = "Z_T60_sine"

bs5 = sagnacf(delta=60)

bs5.load_data(config['path_to_data'], "FJZ_20241023_backscatter_T60_sine.pkl")

bs5.relative_timing(config['tbeg'])

bs5.apply_backscatter_correction()

bs5.trim(tbeg=UTCDateTime("2024-10-23 12:00"))

In [13]:
lbl6 = "Z_T120_sine"

bs6 = sagnacf(delta=120)

bs6.load_data(config['path_to_data'], "FJZ_20241023_backscatter_T120_sine.pkl")

bs6.relative_timing(config['tbeg'])

bs6.apply_backscatter_correction()

bs6.trim(tbeg=UTCDateTime("2024-10-23 12:00"))

In [14]:
lbl7 = "Z_T120_sine_ca"

bs7 = sagnacf(delta=120)

bs7.load_data(config['path_to_data'], "FJZ_20241023_backscatter_T120_sine_CA.pkl")

bs7.relative_timing(config['tbeg'])

bs7.apply_backscatter_correction()

bs7.trim(tbeg=UTCDateTime("2024-10-23 12:00"))

In [15]:
lbl8 = "Z_T60_sine_ca"

bs8 = sagnacf(delta=120)

bs8.load_data(config['path_to_data'], "FJZ_20241023_backscatter_T60_sine_CA.pkl")

bs8.relative_timing(config['tbeg'])

bs8.apply_backscatter_correction()

bs8.trim(tbeg=UTCDateTime("2024-10-23 12:00"))

In [16]:
lbl0 = "Z_T200_hilbert"

bs0 = sagnacf(delta=120)

bs0.load_data(config['path_to_data'], "FJZ_20241023_backscatter_T200_hilbert.pkl")

bs0.relative_timing(config['tbeg'])

bs0.apply_backscatter_correction()

bs0.trim(tbeg=UTCDateTime("2024-10-23 12:00"))

In [17]:
%matplotlib tk

## Compare Hilbert vs. Sine

In [18]:
def __makeplot():

    import matplotlib.pyplot as plt

    Nrow, Ncol = 2, 1

    font = 12

    tscale, tunit = 1/3600, "hour"

    fig, ax = plt.subplots(Nrow, Ncol, figsize=(12, 5), sharex=True)

    plt.subplots_adjust(hspace=0.1)


    ax[0].plot(bs4.data.time_sec*tscale, bs4.data.fj_fs, label=lbl4)
    ax[0].plot(bs2.data.time_sec*tscale, bs2.data.fj_fs, label=lbl2)
    # ax[0].plot(bs1.data.time_sec*tscale, bs1.data.fj_fs, label=lbl1)
    # ax[0].plot(bs3.data.time_sec*tscale, bs3.data.fj_fs, label=lbl3)
    # ax[0].plot(bs6.data.time_sec*tscale, bs6.data.fj_fs, label=lbl6)
    ax[0].plot(bs5.data.time_sec*tscale, bs5.data.fj_fs, label=lbl5)
    # ax[0].plot(bs7.data.time_sec*tscale, bs7.data.fj_fs, label=lbl7)
    ax[0].plot(bs8.data.time_sec*tscale, bs8.data.fj_fs, label=lbl8)


    ax[1].plot(bs4.data.time_sec*tscale, bs4.data.fj_bs, label=lbl4)
    ax[1].plot(bs2.data.time_sec*tscale, bs2.data.fj_bs, label=lbl2)
    # ax[1].plot(bs1.data.time_sec*tscale, bs1.data.fj_bs, label=lbl1)
    # ax[1].plot(bs3.data.time_sec*tscale, bs3.data.fj_bs, label=lbl3)
    # ax[1].plot(bs6.data.time_sec*tscale, bs6.data.fj_bs, label=lbl6)
    ax[1].plot(bs5.data.time_sec*tscale, bs5.data.fj_bs, label=lbl5)
    ax[1].plot(bs8.data.time_sec*tscale, bs8.data.fj_bs, label=lbl8)

    ax[0].legend(ncol=2)

    ax[0].set_ylabel("$\delta$f (Hz) w\ bs ", fontsize=font)
    ax[1].set_ylabel("$\delta$f(Hz) w\o bs ", fontsize=font)

    ax[1].set_xlabel(f"Time ({tunit})", fontsize=font)

    for j in range(Nrow):

        ax[j].grid(which="both", ls="--", color="grey", alpha=0.5, zorder=0)
        ax[j].ticklabel_format(useOffset=False)

    ax[0].set_ylim(553.496, 553.499)
    ax[1].set_ylim(553.496, 553.499)

    for _k, ll in enumerate(['(a)', '(b)']):
        ax[_k].text(0.005, 0.97, ll, ha="left", va="top", transform=ax[_k].transAxes, fontsize=font+1)

    plt.show();
    return fig

fig = __makeplot();

## Compare T60 vs. T120

In [31]:
def __makeplot():

    import matplotlib.pyplot as plt

    Nrow, Ncol = 2, 1

    font = 12

    tscale, tunit = 1/3600, "hour"

    fig, ax = plt.subplots(Nrow, Ncol, figsize=(12, 5), sharex=True)

    plt.subplots_adjust(hspace=0.1)

    ax[0].plot(bs2.data.time_sec*tscale, bs2.data.fj_fs, label=lbl2)
    ax[0].plot(bs1.data.time_sec*tscale, bs1.data.fj_fs, label=lbl1)

    ax[0].plot(bs5.data.time_sec*tscale, bs5.data.fj_fs, label=lbl5)
    ax[0].plot(bs6.data.time_sec*tscale, bs6.data.fj_fs, label=lbl6)

    # ax[0].plot(bs0.data.time_sec*tscale, bs0.data.fj_fs, label=lbl0)

    ax[1].plot(bs2.data.time_sec*tscale, bs2.data.fj_bs, label=lbl2)
    ax[1].plot(bs1.data.time_sec*tscale, bs1.data.fj_bs, label=lbl1)

    ax[1].plot(bs5.data.time_sec*tscale, bs5.data.fj_bs, label=lbl5)
    ax[1].plot(bs6.data.time_sec*tscale, bs6.data.fj_bs, label=lbl6)

    # ax[1].plot(bs0.data.time_sec*tscale, bs0.data.fj_bs, label=lbl0)

    ax[0].legend(ncol=2)

    ax[0].set_ylabel("$\delta$f (Hz) w\ bs ", fontsize=font)
    ax[1].set_ylabel("$\delta$f(Hz) w\o bs ", fontsize=font)

    ax[1].set_xlabel(f"Time ({tunit})", fontsize=font)

    for j in range(Nrow):

        ax[j].grid(which="both", ls="--", color="grey", alpha=0.5, zorder=0)
        ax[j].ticklabel_format(useOffset=False)

    ax[0].set_ylim(553.496, 553.499)
    ax[1].set_ylim(553.496, 553.499)

    for _k, ll in enumerate(['(a)', '(b)']):
        ax[_k].text(0.005, 0.97, ll, ha="left", va="top", transform=ax[_k].transAxes, fontsize=font+1)

    plt.show();
    return fig

fig = __makeplot();

## Compare Frequency Band

In [20]:
lbl10 = "Z_T60_hilbert_2Hz"

bs10 = sagnacf(delta=60)

bs10.load_data(config['path_to_data'], "FJZ_20241023_backscatter_T60_hilbert.pkl")

bs10.relative_timing(config['tbeg'])

bs10.apply_backscatter_correction()

bs10.trim(tbeg=UTCDateTime("2024-10-23 12:00"))

In [21]:
lbl11 = "Z_T200_hilbert_5Hz"

bs11 = sagnacf(delta=60)

bs11.load_data(config['path_to_data'], "FJZ_20241023_backscatter_T60_hilbert5Hz.pkl")

bs11.relative_timing(config['tbeg'])

bs11.apply_backscatter_correction()

bs11.trim(tbeg=UTCDateTime("2024-10-23 12:00"))

In [22]:
lbl12 = "Z_T200_hilbert_1Hz"

bs12 = sagnacf(delta=60)

bs12.load_data(config['path_to_data'], "FJZ_20241023_backscatter_T60_hilbert1Hz.pkl")

bs12.relative_timing(config['tbeg'])

bs12.apply_backscatter_correction()

bs12.trim(tbeg=UTCDateTime("2024-10-23 12:00"))

## Sampling as high as 1Hz

In [23]:
def __makeplot():

    import matplotlib.pyplot as plt

    Nrow, Ncol = 2, 1

    font = 12

    tscale, tunit = 1/3600, "hour"

    fig, ax = plt.subplots(Nrow, Ncol, figsize=(12, 5), sharex=True)

    plt.subplots_adjust(hspace=0.1)

    ax[0].plot(bs11.data.time_sec*tscale, bs11.data.fj_fs, label=lbl11)
    ax[0].plot(bs10.data.time_sec*tscale, bs10.data.fj_fs, label=lbl10)
    ax[0].plot(bs12.data.time_sec*tscale, bs12.data.fj_fs, label=lbl12)

    ax[1].plot(bs11.data.time_sec*tscale, bs11.data.fj_bs, label=lbl11)
    ax[1].plot(bs10.data.time_sec*tscale, bs10.data.fj_bs, label=lbl10)
    ax[1].plot(bs12.data.time_sec*tscale, bs12.data.fj_bs, label=lbl12)


    ax[0].legend(ncol=2)

    ax[0].set_ylabel("$\delta$f (Hz) w\ bs ", fontsize=font)
    ax[1].set_ylabel("$\delta$f(Hz) w\o bs ", fontsize=font)

    ax[1].set_xlabel(f"Time ({tunit})", fontsize=font)

    for j in range(Nrow):

        ax[j].grid(which="both", ls="--", color="grey", alpha=0.5, zorder=0)
        ax[j].ticklabel_format(useOffset=False)

    ax[0].set_ylim(553.496, 553.499)
    ax[1].set_ylim(553.496, 553.499)

    for _k, ll in enumerate(['(a)', '(b)']):
        ax[_k].text(0.005, 0.97, ll, ha="left", va="top", transform=ax[_k].transAxes, fontsize=font+1)

    plt.show();
    return fig

fig = __makeplot();

In [70]:
lblx = "Z_T200_hilbert_1Hz"

bsx = sagnacf(delta=60)

# bsx.load_data("/import/kilauea-data/sagnac_frequency/data/backscatter/",
#               "FJZ_20241023_backscatter_T10_hilbert.pkl"
#              )

bsx.load_data("/import/kilauea-data/sagnac_frequency/data/compare_methods/backscatter/",
              "FJZ_20241023_backscatter_T60_hilbert_CA_testrun_10.pkl"
             )


bsx.relative_timing(config['tbeg'])

bsx.apply_backscatter_correction()

bsx.trim(tbeg=UTCDateTime("2024-10-23 00:00"))

In [71]:
plt.plot(bsx.data.time_sec, bsx.data.fj_fs)

In [72]:
# lblx = "Z_T200_hilbert_1Hz"

bsz = sagnacf(delta=10)

bsz.load_data("/import/kilauea-data/sagnac_frequency/data/compare_methods/backscatter/",
              "FJZ_20241023_backscatter_T10_hilbert_CA_testrun_10.pkl"
             )

bsz.relative_timing(config['tbeg'])

bsz.apply_backscatter_correction()

bsz.trim(tbeg=UTCDateTime("2024-10-23 00:00"))

In [73]:
# plt.plot(bsx.data.time_sec, bsx.data.fj_fs)
plt.plot(bsz.data.time_sec, bsz.data.fj_fs)